# Article Visualizations

Six publication figures from `atlanta.ipynb` and `chicago.ipynb`, written to
`data/processed/viz/`.

## How to read this notebook

Every figure follows the same four moves. Once you see the pattern, the rest is
detail:

1. **`fig, ax = plt.subplots(...)`** — make a canvas and get a handle on it
2. **`ax.<something>(...)`** — draw onto that handle (`bar`, `plot`, `scatter`, `.plot(ax=ax)`)
3. **`ax.set_*` / `fig.text`** — label it
4. **`plt.savefig(...)`** — write it out

The design decisions matter more than the syntax, so those are the comments
worth reading. The three rules doing the most work here:

- **One hue for magnitude.** A sequential ramp (`Blues`) encodes "how much" through
  *lightness*, which survives colour-vision deficiency and greyscale printing.
  Rainbow ramps like `jet` do not, and `viridis` is multi-hue.
- **Two hues for identity.** Atlanta orange vs Chicago blue — far apart in both hue
  and lightness, so they never collapse into each other.
- **Label the marks, not the grid.** A number printed on a bar beats five gridlines.

## Setup

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy import stats
from shapely import wkt
from pathlib import Path

warnings.filterwarnings("ignore")

REPO = Path("/Users/medhasubramaniyan/Desktop/GitHub_Repos/hoop-deserts-atl")
OUT  = REPO / "data/processed/viz"
OUT.mkdir(parents=True, exist_ok=True)

## The design system

Defining colours and rcParams once, up front, is what makes six figures look like
one set instead of six one-offs. `plt.rcParams` is matplotlib's global stylesheet —
set it here and every later figure inherits it.

In [ ]:
# Neutrals. Text is near-black rather than pure #000 (softer in print);
# the grid is barely-there so it never competes with the data.
INK, MUTED, GRID, SURF = "#1a1a1a", "#6b6b6b", "#e5e5e3", "#ffffff"

# Identity colours: one per city. Chosen far apart in hue AND lightness so they
# stay distinguishable under protanopia/deuteranopia and in greyscale.
ATL, CHI = "#e8833a", "#2a6fb5"
COURT = "#d64545"        # court dots on maps

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,   # drop chart junk
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
})

SRC = "OpenStreetMap · CDC PLACES 2025 · ACS 2023 5-yr"
QL = ["Q1\nlowest", "Q2", "Q3", "Q4\nhighest"]
LAB = {"Atlanta": "Atlanta  (Fulton + DeKalb)", "Chicago": "Chicago  (city limits)"}

def tag(fig):
    """Source line, bottom-left of every figure. Figures get separated from their
    article, so each one has to carry its own provenance."""
    fig.text(0.01, 0.012, SRC, fontsize=7.5, color=MUTED, ha="left")

def drop_na_q(a):
    """income_q is a Categorical; casting to str turns missing income into the
    literal string 'nan', which would otherwise plot as a 5th quartile."""
    return a[a.income_q != "nan"]

## Loading both cities

Rather than duplicate the pipeline, this executes each notebook's code cells and
lifts `analysis` and `courts` out of the resulting namespace. Slightly unusual,
but it guarantees the figures never drift from the analysis.

Two things happen here that affect the numbers:

**Race shares.** `pct_white`, `pct_black`, `pct_hispanic` are computed for all
three groups. ACS race (B02001) and Hispanic origin (B03003) are *separate*
universes — a Hispanic resident is also counted in a race category — so these are
three independent shares and do **not** sum to 100%.

**Chicago is restricted to city limits.** Its Overpass export only ever covered
the city, so drawing it at Cook County extent makes 551 unqueried suburban tracts
look court-free. 97.5% of mapped courts are inside the city, so the city is the
honest unit — and it matches the article's argument, since recreation policy is
set at city level.

In [ ]:
def load_city(name, nb_path):
    g = {}
    for cell in json.loads(Path(nb_path).read_text())["cells"]:
        if cell["cell_type"] == "code":
            try:
                exec("".join(cell["source"]), g)
            except Exception:
                pass          # plotting cells may fail headless; data cells matter
    a, courts = g["analysis"].copy(), g["courts"].copy()

    for new, src in [("pct_white", "pop_white_alone"),
                     ("pct_black", "pop_black_alone"),
                     ("pct_hispanic", "pop_hispanic_or_latino")]:
        a[new] = a[src] / a.race_universe * 100

    # Attach neighborhood names via centroid-in-polygon.
    if name == "Atlanta":
        hd = gpd.read_file(REPO/"data/raw/Official_Neighborhoods_-_Open_Data.geojson")
        col = "NAME"
    else:
        h = pd.read_csv(REPO/"data/raw/chicago_hoods.csv")
        h["geometry"] = h["the_geom"].apply(wkt.loads)   # CSV holds WKT, not a spatial file
        hd = gpd.GeoDataFrame(h, geometry="geometry", crs="EPSG:4326")
        col = "PRI_NEIGH"
    hd = hd[[col, "geometry"]].to_crs(a.crs)

    pts = a[["GEOID", "geometry"]].copy()
    pts["geometry"] = pts.geometry.centroid
    lab = (gpd.sjoin(pts, hd, how="left", predicate="within")
             .drop(columns="index_right").rename(columns={col: "neighborhood"}))
    a = a.merge(lab[["GEOID", "neighborhood"]], on="GEOID", how="left")
    a["income_q"] = a["income_q"].astype(str)

    # Chicago: keep only tracts the court export actually covers.
    if name == "Chicago":
        keep = set(a.loc[a.neighborhood.notna(), "GEOID"])
        a = a[a.GEOID.isin(keep)].copy()
        courts = courts[courts.GEOID.isin(keep)].copy()
    return a, courts

R = {n: load_city(n, REPO/f"notebooks/{n.lower()}.ipynb")
     for n in ["Atlanta", "Chicago"]}

for n, (a, c) in R.items():
    print(f"{n}: {len(a)} tracts, {len(c)} courts")

## Figure 1 — Courts by income quartile

**The move:** two panels sharing a y-axis, so bar heights are directly comparable
across cities. `sharey=True` is what makes that true; without it matplotlib scales
each panel independently and the comparison silently breaks.

**Why per 10,000 and not raw counts:** quartiles hold different populations, so
raw counts would mostly measure population. Note the denominator is computed on
the *pooled* quartile (`d.court_count.sum() / (d.total_population.sum()/1e4)`),
not as a mean of tract-level rates — with 77–81% of tracts at zero, a mean of
rates gets dragged around by a handful of tiny-population tracts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharey=True)

for ax, (city, color) in zip(axes, [("Atlanta", ATL), ("Chicago", CHI)]):
    a = drop_na_q(R[city][0])
    q = (a.groupby("income_q", observed=True)
           .apply(lambda d: d.court_count.sum() / (d.total_population.sum()/1e4),
                  include_groups=False)
           .reindex(sorted(a.income_q.unique())))

    ax.bar(range(4), q.values, color=color, width=0.62, zorder=3)

    # Direct labels instead of dense gridlines — the reader gets the number
    # without tracing back to an axis.
    for i, v in enumerate(q.values):
        ax.text(i, v + 0.05, f"{v:.2f}", ha="center",
                fontsize=11, fontweight="bold", color=INK)

    ax.set_xticks(range(4)); ax.set_xticklabels(QL, fontsize=9.5)
    ax.set_title(LAB[city], fontsize=13, fontweight="bold", loc="left", color=INK, pad=10)
    ax.grid(axis="y", color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)      # grid behind bars, not through them
    ax.set_ylim(0, 2.45)

axes[0].set_ylabel("courts per 10,000 residents", fontsize=10)

# fig.suptitle/fig.text position in FIGURE coordinates (0-1), independent of axes.
fig.suptitle("Chicago built courts where income is lowest. Atlanta built them at both ends.",
             fontsize=15.5, fontweight="bold", x=.01, ha="left", y=.99, color=INK)
fig.text(.01, .905, "Basketball courts per 10,000 residents, by census-tract income quartile",
         fontsize=10.5, color=MUTED, ha="left")

# rect reserves space so tight_layout doesn't overlap the title block.
plt.tight_layout(rect=[0, .035, 1, .88]); tag(fig)
plt.savefig(OUT/"01_courts_by_income_quartile.png", dpi=220, bbox_inches="tight")
plt.show()

## Figure 2 — Inactivity slope

**The move:** a slope chart. Four categories, two series — a line makes the
*trend* the subject, where grouped bars would make individual values the subject.

Endpoint labels sit outside the plot area (`x = -0.13` and `3.13`, past the 0–3
data range) with `set_xlim` widened to make room. That's why there's no legend
clutter competing with the lines.

**Median, not mean** — inactivity is skewed, and the median is the resistant
summary.

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 6))

for city, color in [("Atlanta", ATL), ("Chicago", CHI)]:
    a = drop_na_q(R[city][0])
    v = (a.groupby("income_q", observed=True)["LPA"].median()
           .reindex(sorted(a.income_q.unique())))

    ax.plot(range(4), v.values, "-o", color=color, lw=2.6, ms=9, zorder=3, label=city)

    # Endpoint labels, pushed just outside the data range.
    ax.text(-0.13, v.values[0],  f"{v.values[0]:.1f}%",  ha="right", va="center",
            fontsize=11, fontweight="bold", color=color)
    ax.text(3.13,  v.values[-1], f"{v.values[-1]:.1f}%", ha="left",  va="center",
            fontsize=11, fontweight="bold", color=color)

ax.set_xticks(range(4)); ax.set_xticklabels(QL, fontsize=10)
ax.set_xlim(-0.75, 3.75)            # headroom for the endpoint labels
ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_ylabel("% adults physically inactive (median)", fontsize=10)
ax.legend(frameon=False, fontsize=11, loc="upper right")

fig.suptitle("The gap that is real, in both cities",
             fontsize=16, fontweight="bold", x=.02, ha="left", y=.99, color=INK)
fig.text(.02, .925,
         "Physical inactivity falls steeply with income — regardless of where courts were built",
         fontsize=10.5, color=MUTED, ha="left")

plt.tight_layout(rect=[0, .035, 1, .90]); tag(fig)
plt.savefig(OUT/"02_inactivity_slope.png", dpi=220, bbox_inches="tight")
plt.show()

## Figure 3 — Correlation dot plot

**The move:** a dot plot, not bars. Correlations are points on a scale, not
magnitudes growing from zero — bars would imply the latter.

**The shaded band is the honest part.** ρ between −0.1 and +0.1 is negligible
regardless of p-value, and with n≈1,300 (Chicago) trivial effects reach
significance easily. Drawing the band means a reader can't mistake "statistically
significant" for "matters."

**Why Spearman:** with 77–81% of tracts at zero and a long right tail, Pearson gets
dragged by a few extreme values. In Atlanta it reported +0.157 (p=0.0003) against
poverty where Spearman found −0.001 (p=0.98) — opposite signs. Rank-based is the
defensible choice.

The `y + off` trick (`±0.17`) dodges the two cities' dots apart on a shared row.

In [ ]:
VARS = [("median_household_income", "Median household income"),
        ("OBESITY", "Obesity"), ("LPA", "Physical inactivity"),
        ("poverty_rate", "Poverty rate"), ("pct_black", "% Black residents"),
        ("pct_hispanic", "% Hispanic residents")]

fig, ax = plt.subplots(figsize=(9.6, 5.0))
y = np.arange(len(VARS))[::-1]          # reversed: first variable at top

ax.axvspan(-0.1, 0.1, color="#f4f4f2", zorder=0)   # the "negligible" band

for city, color, off in [("Atlanta", ATL, 0.17), ("Chicago", CHI, -0.17)]:
    a = R[city][0]
    xs = [stats.spearmanr(*a[["courts_per_10k", v]].dropna().values.T).statistic
          for v, _ in VARS]
    ax.scatter(xs, y + off, s=125, color=color, zorder=3, label=city,
               edgecolor=SURF, lw=1.4)   # white ring keeps overlapping dots readable

ax.axvline(0, color=INK, lw=1.1, zorder=2)
ax.text(0, len(VARS) - 0.42, "negligible", ha="center", fontsize=8.5,
        color=MUTED, style="italic")

ax.set_yticks(y); ax.set_yticklabels([l for _, l in VARS], fontsize=10.5, color=INK)
ax.set_xlim(-0.34, 0.34); ax.set_ylim(-0.75, len(VARS) - 0.25)
ax.set_xlabel("Spearman ρ  —  court density vs. tract characteristic", fontsize=10)
ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=11, loc="lower left",
          bbox_to_anchor=(0, -0.015), ncol=2)

fig.suptitle("Court density predicts nothing in Atlanta — and the reverse of the\n"
             "hoop-desert theory in Chicago",
             fontsize=15, fontweight="bold", x=.01, ha="left", y=1.0, color=INK)

plt.tight_layout(rect=[0, .035, 1, .88]); tag(fig)
plt.savefig(OUT/"03_correlations.png", dpi=220, bbox_inches="tight")
plt.show()

## Figure 4 — Race composition by quartile

**The move:** grouped bars, deliberately *not* stacked. ACS race and Hispanic
origin are separate universes, so the three shares don't sum to 100% — stacking
would draw a whole that doesn't exist. The subtitle says so explicitly.

Grouping is done by offsetting x positions by the bar width: `x-w`, `x`, `x+w`.

**Why this figure earns its place:** it shows Chicago's Q2 is the Hispanic
quartile (30.5% median) — and court density there has already dropped from 2.03
to 1.46. Courts correlate +0.21 with pct_black but −0.07 with pct_hispanic. A
pct_black-only analysis would have rendered Little Village and New City — two of
the highest-inactivity neighborhoods *with* courts — as unexplained noise.

In [ ]:
CW, CB, CH = "#8c8c8c", "#4a3aa7", "#1baf7a"     # White / Black / Hispanic

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, city in zip(axes, ["Atlanta", "Chicago"]):
    a = drop_na_q(R[city][0])
    g = (a.groupby("income_q", observed=True)[["pct_white", "pct_black", "pct_hispanic"]]
           .median().reindex(sorted(a.income_q.unique())))

    x, w = np.arange(4), 0.26
    ax.bar(x - w, g.pct_white,    w, color=CW, label="White",    zorder=3)
    ax.bar(x,     g.pct_black,    w, color=CB, label="Black",    zorder=3)
    ax.bar(x + w, g.pct_hispanic, w, color=CH, label="Hispanic", zorder=3)

    ax.set_xticks(x); ax.set_xticklabels(QL, fontsize=9.5)
    ax.set_title(LAB[city], fontsize=13, fontweight="bold", loc="left", color=INK, pad=10)
    ax.grid(axis="y", color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True); ax.set_ylim(0, 95)

axes[0].set_ylabel("median % of tract population", fontsize=10)
axes[0].legend(frameon=False, fontsize=10, ncol=3, loc="upper center")

fig.suptitle('"Low income" is not one population — and the courts followed only one of them',
             fontsize=15, fontweight="bold", x=.01, ha="left", y=.99, color=INK)
fig.text(.01, .905, "Median race/ethnicity share by income quartile. ACS race and Hispanic "
         "origin are separate measures, so shares do not sum to 100%.",
         fontsize=10, color=MUTED, ha="left")

plt.tight_layout(rect=[0, .035, 1, .87]); tag(fig)
plt.savefig(OUT/"04_race_by_quartile.png", dpi=220, bbox_inches="tight")
plt.show()

## Figure 5 — Hero maps (2×2)

**The move:** dots on the left, choropleth on the right. That split is a
deliberate choice, not a stylistic one.

**Why courts are dots, not a choropleth.** 77–81% of tracts have zero courts and
the max is ~34 per 10k. A colour ramp on that distribution renders nearly every
tract near-white with a few dark specks — it *looks* like a pattern exists at a
resolution the data can't support. Dots show location honestly.

**Layering** is `ax=ax`: `.plot()` returns an Axes, and passing that same handle
to the next `.plot()` call draws onto it. Order = z-order, so tracts first,
courts second.

**`make_axes_locatable`** solves a real geopandas annoyance: `legend=True` steals
width from the map, so panels end up different sizes. Creating an explicit
colorbar axis (`cax`) keeps both map panels the same width.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 12))

for r, city in enumerate(["Atlanta", "Chicago"]):
    a, courts = R[city]

    # LEFT — court locations on a plain base.
    a.plot(ax=axes[r, 0], color="#f2f2f0", edgecolor="white", linewidth=.2)
    courts.plot(ax=axes[r, 0], color=COURT, markersize=8, alpha=.85)
    axes[r, 0].set_title(f"{LAB[city]}   ·   {len(courts)} courts",
                         fontsize=12, fontweight="bold", loc="left", color=INK, pad=6)

    # RIGHT — inactivity choropleth, single-hue sequential ramp.
    div = make_axes_locatable(axes[r, 1])
    cax = div.append_axes("right", size="3.5%", pad=.1)
    a.plot(ax=axes[r, 1], column="LPA", cmap="Blues", legend=True, cax=cax,
           edgecolor="white", linewidth=.1,
           missing_kwds={"color": "#dddddd"})    # grey = no PLACES estimate, not zero
    cax.tick_params(labelsize=8, colors=MUTED)
    cax.set_ylabel("% inactive", fontsize=8.5, color=MUTED)
    axes[r, 1].set_title(f"{LAB[city]}   ·   inactivity {a.LPA.min():.0f}%–{a.LPA.max():.0f}%",
                         fontsize=12, fontweight="bold", loc="left", color=INK, pad=6)

    for c in range(2):
        axes[r, c].set_axis_off()    # lat/lon ticks mean nothing to a reader

fig.suptitle("Courts are scattered. Inactivity is sorted.",
             fontsize=19, fontweight="bold", x=.02, ha="left", y=.985, color=INK)
fig.text(.02, .952, "Left: every mapped basketball court.  Right: share of adults "
         "reporting no physical activity outside work.", fontsize=11, color=MUTED, ha="left")
fig.text(.02, .008, SRC, fontsize=8, color=MUTED, ha="left")

plt.tight_layout(rect=[0, .018, 1, .94])
plt.savefig(OUT/"05_hero_maps.png", dpi=190, bbox_inches="tight")
plt.show()

## Figure 6 — Tag coverage

**The move:** horizontal bars, because the category labels are words. Vertical
bars would force rotated labels, which are harder to read.

This figure argues an *absence*. Lighting decides whether a court is usable after
work in winter — the single attribute most likely to explain who actually plays —
and it's recorded on under 10% of courts in both cities. The subtitle gives raw
counts so "9%" doesn't sound like a sample rather than a near-total gap.

In [ ]:
TAGS = [("lit", "Lighting"), ("surface", "Surface type"),
        ("access", "Public/private"), ("hoops", "Number of hoops")]

fig, ax = plt.subplots(figsize=(9.6, 4.8))
y = np.arange(len(TAGS))[::-1]

for city, color, off in [("Atlanta", ATL, 0.17), ("Chicago", CHI, -0.17)]:
    courts = R[city][1]
    vals = [courts[t].notna().mean()*100 if t in courts.columns else 0 for t, _ in TAGS]
    ax.barh(y + off, vals, height=0.32, color=color, label=city, zorder=3)
    for yy, vv in zip(y + off, vals):
        ax.text(vv + 1.2, yy, f"{vv:.0f}%", va="center",
                fontsize=9.5, color=INK, fontweight="bold")

ax.set_yticks(y); ax.set_yticklabels([l for _, l in TAGS], fontsize=11, color=INK)
ax.set_xlim(0, 42)
ax.set_xlabel("% of mapped courts where the attribute is recorded", fontsize=10, color=MUTED)
ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.spines["left"].set_color(GRID); ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED)
ax.legend(frameon=False, fontsize=10.5, loc="lower right")

a_lit, a_n = R["Atlanta"][1].lit.notna().sum(), len(R["Atlanta"][1])
c_lit, c_n = R["Chicago"][1].lit.notna().sum(), len(R["Chicago"][1])

fig.suptitle("The thing most likely to explain who plays is the thing nobody records",
             fontsize=15, fontweight="bold", x=.01, ha="left", y=.99, color=INK)
fig.text(.01, .90, f"Lighting determines whether a court is usable after work in winter. "
         f"It is recorded on {a_lit} of {a_n} Atlanta courts\nand {c_lit} of {c_n} in Chicago.",
         fontsize=10, color=MUTED, ha="left")

plt.tight_layout(rect=[0, .035, 1, .85]); tag(fig)
plt.savefig(OUT/"06_tag_coverage.png", dpi=220, bbox_inches="tight")
plt.show()

## Headline numbers

Printed here so the figures and the article prose can't drift apart. Any number
you write should come from this cell.

In [ ]:
for city in ["Atlanta", "Chicago"]:
    a = drop_na_q(R[city][0])
    q1 = a[a.income_q.str.startswith("Q1")]
    q4 = a[a.income_q.str.startswith("Q4")]
    r1 = q1.court_count.sum()/(q1.total_population.sum()/1e4)
    r4 = q4.court_count.sum()/(q4.total_population.sum()/1e4)
    m = a[["courts_per_10k", "median_household_income"]].dropna()
    rho = stats.spearmanr(m.courts_per_10k, m.median_household_income).statistic

    print(f"\n{city} — {len(R[city][0])} tracts, {len(R[city][1])} courts")
    print(f"  Q1 {r1:.2f} vs Q4 {r4:.2f} courts/10k  (ratio {r1/r4:.2f})")
    print(f"  Spearman rho, density vs income: {rho:+.3f}")
    print(f"  inactivity Q1 {q1.LPA.median():.1f}% -> Q4 {q4.LPA.median():.1f}%")
    print(f"  tracts with no court: {(R[city][0].court_count == 0).mean():.0%}")